# 국민연금 2027 목표비중 — 통합 재현 노트북

이 노트북은 **고정된 2016-09~2026-08 프록시 월수익률 입력**에서 시작하여 STEP 2부터 최종 제출 CSV까지 재생성한다. 핵심 실행에는 yfinance 네트워크 접속이 필요하지 않는다.

- Yahoo Finance는 고정 프록시 가격자료의 원출처로 공개한다.
- official mapped의 DM/EM 92:8 및 대체 1/3씩은 CMA와 독립적인 팀 중립 매핑 규칙이다.
- EM/PE/인프라/PD 10% 상한은 공식 NPS 한도가 아니라 팀의 세부 집중도 가정이다.
- 현금 0.1% 고정을 baseline으로 하고 0~2% 최적화는 sensitivity로 별도 계산한다.
- Table B의 Michaud는 순수 resampling이며 LW+Michaud는 sensitivity다.


In [1]:
from pathlib import Path
import runpy, json
import pandas as pd
from IPython.display import display

HERE=Path.cwd().resolve()
if (HERE/"src").exists():
    ROOT=HERE
elif (HERE.parent/"src").exists():
    ROOT=HERE.parent
else:
    raise FileNotFoundError("Run from repository root or submission/ directory.")

RESULTS=ROOT/"results"
SUBMISSION=ROOT/"submission"
RESULTS.mkdir(exist_ok=True); SUBMISSION.mkdir(exist_ok=True)

def show_csv(rel,n=30):
    p=ROOT/rel
    print("\n",rel)
    df=pd.read_csv(p)
    display(df.head(n))
    return df

print("ROOT =",ROOT)


ROOT = /home/runner/work/fund/fund


## STEP 2 — 고정 프록시 수익률 → 상관 → CMA 공분산

원자료는 Yahoo Finance에서 추출한 프록시 가격을 기반으로 하지만, 분석 실행은 저장소에 고정된 월수익률을 사용한다.


In [2]:
runpy.run_path(str(ROOT/"src/step2_build_corr.py"),run_name="__main__")
show_csv("data/diagnostics.csv")


Wrote /home/runner/work/fund/fund/data/team2_step2_corr_package.xlsx

 data/diagnostics.csv


,check,value
0,T_monthly,120
1,start_month,2016-09-30
2,end_month,2026-08-31
3,corr_symmetry_max_abs_error,0.0
4,corr_diag_max_abs_error,0.0
5,corr_min_eigenvalue,0.09005407607642206
6,corr_condition_number,47.995136506497644
7,cov_min_eigenvalue,0.0014933309924158925
8,cov_condition_number,80.84833239095349
9,cov_diag_sigma_max_abs_error,0.0


,check,value
0,T_monthly,120
1,start_month,2016-09-30
2,end_month,2026-08-31
3,corr_symmetry_max_abs_error,0.0
4,corr_diag_max_abs_error,0.0
5,corr_min_eigenvalue,0.09005407607642206
6,corr_condition_number,47.995136506497644
7,cov_min_eigenvalue,0.0014933309924158925
8,cov_condition_number,80.84833239095349
9,cov_diag_sigma_max_abs_error,0.0


## STEP 5~8 — 기준 MVO, Ledoit–Wolf, Box, Ellipsoid


In [3]:
runpy.run_path(str(ROOT/"src/analyze_steps5_8.py"),run_name="__main__")
show_csv("results/step5_mvo.csv")
show_csv("results/step6_lw_diagnostics.csv")
show_csv("results/step7_box.csv")
show_csv("results/step8_ellipsoid.csv")


LW shrinkage: 0.12191206329292961
LW weights: {'국내주식': np.float64(0.1998718639397802), '글로벌 선진국 DM': np.float64(0.22012813606021986), '글로벌 신흥국 EM': np.float64(0.1), '국내 국채(초장기)': np.float64(0.18000000000000002), '글로벌 IG 크레딧': np.float64(0.11999999999999998), '사모주식 PE/VC': np.float64(1.3652245191626905e-17), '실물 인프라': np.float64(0.07999999999999996), '사모대출 PD': np.float64(0.09999999999999996)}
LW metrics: {'expected_return': 0.05919923118363868, 'volatility': 0.0953608032515817, 'sharpe': 0.30619741222821933, 'TE': 0.01678942061220061, 'turnover': 0.20198865532198856}


Saved results to /home/runner/work/fund/fund/results

 results/step5_mvo.csv


,구분,gamma,국내주식,글로벌 선진국 DM,글로벌 신흥국 EM,국내 국채(초장기),글로벌 IG 크레딧,사모주식 PE/VC,실물 인프라,사모대출 PD,기대수익률,변동성,Sharpe,TE,회전율
0,단순 MVO,2.0,5.106592e-17,7.068998e-17,9.946723e-02,1.190454e-16,0.000000,3.615814e-17,6.418477e-17,0.900533,0.078597,0.096243,0.504939,0.074975,0.923777
1,정책 MVO,2.0,2.200000e-01,2.000000e-01,1.000000e-01,1.800000e-01,0.120000,4.617835e-18,8.000000e-02,0.100000,0.059320,0.101177,0.289789,0.018617,0.213780
2,단순 MVO,4.0,5.541935e-03,6.672575e-17,5.412805e-02,2.613523e-17,0.000000,1.817289e-17,0.000000e+00,0.940330,0.078236,0.094972,0.507897,0.079225,0.918235
3,정책 MVO,4.0,2.137636e-01,2.062364e-01,1.000000e-01,1.800000e-01,0.120000,8.687312e-17,8.000000e-02,0.100000,0.059283,0.101078,0.289702,0.017992,0.207544
4,단순 MVO,6.0,5.160768e-02,0.000000e+00,2.456480e-17,4.785456e-02,0.125971,0.000000e+00,2.262425e-17,0.774567,0.072015,0.081061,0.518315,0.071514,0.778749
5,정책 MVO,6.0,1.884284e-01,2.315716e-01,1.000000e-01,2.300000e-01,0.120000,0.000000e+00,3.000000e-02,0.100000,0.057531,0.097898,0.281216,0.016983,0.181485



 results/step6_lw_diagnostics.csv


,구분,최소고유값,조건수
0,표본 공분산,0.001492,91.849310
1,Ledoit-Wolf,0.004893,25.320212
2,CMA Sigma,0.001493,80.848332
3,LW 상관+CMA sigma,0.002839,38.621099



 results/step7_box.csv


,자산,CMA_mu,Box폭,최악_mu,Box비중
0,국내주식,0.062,0.010,0.052,0.206458
1,글로벌 선진국 DM,0.056,0.010,0.046,0.213542
2,글로벌 신흥국 EM,0.084,0.010,0.074,0.100000
3,국내 국채(초장기),0.036,0.005,0.031,0.216517
4,글로벌 IG 크레딧,0.053,0.005,0.048,0.120000
5,사모주식 PE/VC,0.068,0.015,0.053,0.000000
6,실물 인프라,0.068,0.015,0.053,0.043483
7,사모대출 PD,0.078,0.015,0.063,0.100000



 results/step8_ellipsoid.csv


,p,kappa,국내주식,글로벌 선진국 DM,글로벌 신흥국 EM,국내 국채(초장기),글로벌 IG 크레딧,사모주식 PE/VC,실물 인프라,사모대출 PD,기대수익률,변동성,Sharpe,TE,회전율,portfolio_mu_SE,worst_mu
0,0.1,1.868031,0.15,0.27,0.1,0.25,0.1,0.000000e+00,0.084847,0.045153,0.056412,0.098523,0.268074,0.015775,0.166332,0.032049,-0.003457
1,0.5,2.710004,0.15,0.27,0.1,0.25,0.1,7.112366e-17,0.100000,0.030000,0.056260,0.098883,0.265566,0.016102,0.181485,0.031902,-0.030195
2,0.9,3.655348,0.15,0.27,0.1,0.25,0.1,0.000000e+00,0.100000,0.030000,0.056260,0.098883,0.265566,0.016102,0.181485,0.031902,-0.060353


,p,kappa,국내주식,글로벌 선진국 DM,글로벌 신흥국 EM,국내 국채(초장기),글로벌 IG 크레딧,사모주식 PE/VC,실물 인프라,사모대출 PD,기대수익률,변동성,Sharpe,TE,회전율,portfolio_mu_SE,worst_mu
0,0.1,1.868031,0.15,0.27,0.1,0.25,0.1,0.000000e+00,0.084847,0.045153,0.056412,0.098523,0.268074,0.015775,0.166332,0.032049,-0.003457
1,0.5,2.710004,0.15,0.27,0.1,0.25,0.1,7.112366e-17,0.100000,0.030000,0.056260,0.098883,0.265566,0.016102,0.181485,0.031902,-0.030195
2,0.9,3.655348,0.15,0.27,0.1,0.25,0.1,0.000000e+00,0.100000,0.030000,0.056260,0.098883,0.265566,0.016102,0.181485,0.031902,-0.060353


## STEP 9 — 순수 Michaud 300회

Table B baseline은 각 draw의 **표본상관 × CMA σ**를 사용하는 순수 Michaud다. LW+Michaud는 별도 sensitivity로 저장한다.


In [4]:
runpy.run_path(str(ROOT/"src/step9_michaud.py"),run_name="__main__")
show_csv("results/step9_michaud_summary.csv")
show_csv("results/step9_michaud_lw_sensitivity.csv")


            baseline_mvo  michaud_mean   p05   p50    p95   std  \
국내주식               21.38         20.09  15.0  22.0  25.00  4.62   
글로벌 선진국 DM         20.62         26.19  20.0  27.0  37.05  6.22   
글로벌 신흥국 EM         10.00          7.47   0.0  10.0  10.00  4.19   
국내 국채(초장기)         18.00         20.27  15.0  21.0  25.00  4.01   
글로벌 IG 크레딧         12.00         10.23   5.0  12.0  12.00  2.64   
사모주식 PE/VC          0.00          1.78   0.0   0.0   8.00  3.11   
실물 인프라              8.00          6.30   0.0   8.0  10.00  3.76   
사모대출 PD            10.00          7.67   0.0  10.0  10.00  3.64   

            change_vs_baseline  freq_at_zero  freq_at_upper  
국내주식                     -1.29          0.00            NaN  
글로벌 선진국 DM                5.57          0.00            NaN  
글로벌 신흥국 EM               -2.53         21.33          69.67  
국내 국채(초장기)                2.27          0.00            NaN  
글로벌 IG 크레딧               -1.77          0.00            NaN  
사모주식 PE/VC              

,asset,baseline_mvo,michaud_mean,p05,p50,p95,std,change_vs_baseline,freq_at_zero,freq_at_upper
0,국내주식,2.137636e-01,0.200858,1.500000e-01,2.200000e-01,0.2500,0.046239,-0.012906,0.000000,NaN
1,글로벌 선진국 DM,2.062364e-01,0.261949,2.000000e-01,2.700000e-01,0.3705,0.062187,0.055713,0.000000,NaN
2,글로벌 신흥국 EM,1.000000e-01,0.074659,0.000000e+00,1.000000e-01,0.1000,0.041877,-0.025341,0.213333,0.696667
3,국내 국채(초장기),1.800000e-01,0.202735,1.500000e-01,2.100000e-01,0.2500,0.040143,0.022735,0.000000,NaN
4,글로벌 IG 크레딧,1.200000e-01,0.102272,5.000000e-02,1.200000e-01,0.1200,0.026426,-0.017728,0.000000,NaN
5,사모주식 PE/VC,8.687312e-17,0.017801,0.000000e+00,2.006317e-17,0.0800,0.031095,0.017801,0.686667,0.040000
6,실물 인프라,8.000000e-02,0.062977,0.000000e+00,8.000000e-02,0.1000,0.037569,-0.017023,0.166667,0.286667
7,사모대출 PD,1.000000e-01,0.076748,1.608330e-17,1.000000e-01,0.1000,0.036354,-0.023252,0.090000,0.636667



 results/step9_michaud_lw_sensitivity.csv


,asset,baseline_mvo,michaud_mean,p05,p50,p95,std,change_vs_baseline,freq_at_zero,freq_at_upper
0,국내주식,2.137636e-01,0.200477,0.15,2.200000e-01,0.250000,0.046232,-0.013286,0.000000,NaN
1,글로벌 선진국 DM,2.062364e-01,0.260104,0.20,2.700000e-01,0.370500,0.060654,0.053868,0.000000,NaN
2,글로벌 신흥국 EM,1.000000e-01,0.077539,0.00,1.000000e-01,0.100000,0.039740,-0.022461,0.173333,0.720000
3,국내 국채(초장기),1.800000e-01,0.200761,0.15,2.016106e-01,0.250000,0.039887,0.020761,0.000000,NaN
4,글로벌 IG 크레딧,1.200000e-01,0.102580,0.05,1.200000e-01,0.120000,0.026450,-0.017420,0.000000,NaN
5,사모주식 PE/VC,8.687312e-17,0.019983,0.00,2.047749e-17,0.096277,0.033367,0.019983,0.673333,0.050000
6,실물 인프라,8.000000e-02,0.063446,0.00,8.000000e-02,0.100000,0.038251,-0.016554,0.186667,0.306667
7,사모대출 PD,1.000000e-01,0.075109,0.00,1.000000e-01,0.100000,0.036951,-0.024891,0.103333,0.606667


,asset,baseline_mvo,michaud_mean,p05,p50,p95,std,change_vs_baseline,freq_at_zero,freq_at_upper
0,국내주식,2.137636e-01,0.200477,0.15,2.200000e-01,0.250000,0.046232,-0.013286,0.000000,NaN
1,글로벌 선진국 DM,2.062364e-01,0.260104,0.20,2.700000e-01,0.370500,0.060654,0.053868,0.000000,NaN
2,글로벌 신흥국 EM,1.000000e-01,0.077539,0.00,1.000000e-01,0.100000,0.039740,-0.022461,0.173333,0.720000
3,국내 국채(초장기),1.800000e-01,0.200761,0.15,2.016106e-01,0.250000,0.039887,0.020761,0.000000,NaN
4,글로벌 IG 크레딧,1.200000e-01,0.102580,0.05,1.200000e-01,0.120000,0.026450,-0.017420,0.000000,NaN
5,사모주식 PE/VC,8.687312e-17,0.019983,0.00,2.047749e-17,0.096277,0.033367,0.019983,0.673333,0.050000
6,실물 인프라,8.000000e-02,0.063446,0.00,8.000000e-02,0.100000,0.038251,-0.016554,0.186667,0.306667
7,사모대출 PD,1.000000e-01,0.075109,0.00,1.000000e-01,0.100000,0.036951,-0.024891,0.103333,0.606667


## STEP 10 — 방법론 종합비교

모든 방법론의 비교 위험지표는 공통 CMA Σ 기준으로 평가하고, LW 자체 covariance 기준 위험은 보조열로 유지한다.


In [5]:
runpy.run_path(str(ROOT/"src/step10_method_synthesis.py"),run_name="__main__")
show_csv("results/step10_tableB_weights.csv")
show_csv("results/step10_tableB_metrics.csv")
show_csv("results/step10_mu_50bp_sensitivity_summary.csv")
show_csv("results/step10_small_eigenvectors.csv")


            Official mapped  Policy MVO  Ledoit-Wolf  Box Robust  \
국내주식                  20.82       21.38        19.99       20.65   
글로벌 선진국 DM            32.78       20.62        22.01       21.35   
글로벌 신흥국 EM             2.85       10.00        10.00       10.00   
국내 국채(초장기)            21.82       18.00        18.00       21.65   
글로벌 IG 크레딧             7.41       12.00        12.00       12.00   
사모주식 PE/VC             4.77        0.00         0.00        0.00   
실물 인프라                 4.77        8.00         8.00        4.35   
사모대출 PD                4.77       10.00        10.00       10.00   

            Ellipsoid Robust  Michaud mean  
국내주식                    15.0         20.09  
글로벌 선진국 DM              27.0         26.19  
글로벌 신흥국 EM              10.0          7.47  
국내 국채(초장기)              25.0         20.27  
글로벌 IG 크레딧              10.0         10.23  
사모주식 PE/VC               0.0          1.78  
실물 인프라                  10.0          6.30  
사모대출 PD                  3.

,Unnamed: 0,Official mapped,Policy MVO,Ledoit-Wolf,Box Robust,Ellipsoid Robust,Michaud mean
0,국내주식,0.208208,2.137636e-01,0.199872,0.206458,1.500000e-01,0.200858
1,글로벌 선진국 DM,0.327848,2.062364e-01,0.220128,0.213542,2.700000e-01,0.261949
2,글로벌 신흥국 EM,0.028509,1.000000e-01,0.100000,0.100000,1.000000e-01,0.074659
3,국내 국채(초장기),0.218218,1.800000e-01,0.180000,0.216517,2.500000e-01,0.202735
4,글로벌 IG 크레딧,0.074074,1.200000e-01,0.120000,0.120000,1.000000e-01,0.102272
5,사모주식 PE/VC,0.047714,8.687312e-17,0.000000,0.000000,7.112366e-17,0.017801
6,실물 인프라,0.047714,8.000000e-02,0.080000,0.043483,1.000000e-01,0.062977
7,사모대출 PD,0.047714,1.000000e-01,0.100000,0.100000,3.000000e-02,0.076748



 results/step10_tableB_metrics.csv


,method,expected_return,volatility,sharpe,TE,turnover,model_cov_volatility,model_cov_sharpe
0,Official mapped,0.055656,0.107174,0.239383,0.000000,0.000000,0.107174,0.239383
1,Policy MVO,0.059283,0.101078,0.289702,0.017992,0.207544,0.101078,0.289702
2,Ledoit-Wolf,0.059199,0.100903,0.289379,0.016789,0.201989,0.095361,0.306197
3,Box Robust,0.058070,0.098876,0.283892,0.018027,0.169703,0.098876,0.283892
4,Ellipsoid Robust,0.056260,0.098883,0.265566,0.016102,0.181485,0.098883,0.265566
5,Michaud mean,0.057592,0.102651,0.268792,0.010553,0.118645,0.102651,0.268792



 results/step10_mu_50bp_sensitivity_summary.csv


,asset,max_abs_own_weight_change,shock_for_max_own,max_portfolio_L1_change,shock_for_max_L1
0,글로벌 선진국 DM,0.038329,0.005,0.076658,0.005
1,국내주식,0.038329,-0.005,0.076658,-0.005
2,글로벌 신흥국 EM,0.000000,-0.005,0.000000,-0.005
3,국내 국채(초장기),0.000000,-0.005,0.000000,-0.005
4,글로벌 IG 크레딧,0.000000,-0.005,0.000000,-0.005
5,사모주식 PE/VC,0.000000,-0.005,0.000000,-0.005
6,실물 인프라,0.000000,-0.005,0.000000,-0.005
7,사모대출 PD,0.000000,-0.005,0.000000,-0.005



 results/step10_small_eigenvectors.csv


,rank_smallest,eigenvalue,국내주식,글로벌 선진국 DM,글로벌 신흥국 EM,국내 국채(초장기),글로벌 IG 크레딧,사모주식 PE/VC,실물 인프라,사모대출 PD
0,1,0.001493,0.159337,-0.491705,-0.130544,-1.000000,0.772296,0.490649,0.245582,-0.785717
1,2,0.002339,0.232178,-0.530986,-0.033824,-0.079968,0.666750,0.005580,-0.465899,1.000000
2,3,0.003607,-0.143922,0.904707,0.043955,-0.238167,0.459591,-0.183693,-1.000000,-0.275064
3,4,0.005326,-0.064748,-0.274118,0.084982,1.000000,0.562295,0.213454,-0.121727,-0.480491


,rank_smallest,eigenvalue,국내주식,글로벌 선진국 DM,글로벌 신흥국 EM,국내 국채(초장기),글로벌 IG 크레딧,사모주식 PE/VC,실물 인프라,사모대출 PD
0,1,0.001493,0.159337,-0.491705,-0.130544,-1.000000,0.772296,0.490649,0.245582,-0.785717
1,2,0.002339,0.232178,-0.530986,-0.033824,-0.079968,0.666750,0.005580,-0.465899,1.000000
2,3,0.003607,-0.143922,0.904707,0.043955,-0.238167,0.459591,-0.183693,-1.000000,-0.275064
3,4,0.005326,-0.064748,-0.274118,0.084982,1.000000,0.562295,0.213454,-0.121727,-0.480491


## STEP 11 — 독립적인 w2027 Team

Team 목표는 특정 최적화 결과를 복사하지 않고 2026 실제비중, 실행가능성, 대체 비유동성, KRW 기준 해외위험을 추가 판단한다.


In [6]:
runpy.run_path(str(ROOT/"src/step11_team_target.py"),run_name="__main__")
show_csv("results/step11_tableC_detailed.csv")
show_csv("results/step11_tableC_common.csv")
show_csv("results/step11_transition_2026H1_to_team.csv")


Team common target
국내주식    22.0
해외주식    34.0
국내채권    20.5
해외채권     8.5
대체투자    14.9
단기자금     0.1
dtype: float64

Metrics
{
  "expected_return": 0.057087000000000006,
  "volatility": 0.10607109660433538,
  "sharpe": 0.255366455774842,
  "TE_vs_official_mapped": 0.00862535325383138,
  "turnover_vs_official_mapped": 0.08318666666666666
}

 results/step11_tableC_detailed.csv


,Unnamed: 0,Official mapped,Policy MVO,Ledoit-Wolf,Box Robust,Ellipsoid Robust,Michaud mean,Team scenario
0,국내주식,0.208000,2.135499e-01,0.199672,0.206251,1.498500e-01,0.200657,0.220
1,글로벌 선진국 DM,0.327520,2.060301e-01,0.219908,0.213329,2.697300e-01,0.261687,0.275
2,글로벌 신흥국 EM,0.028480,9.990000e-02,0.099900,0.099900,9.990000e-02,0.074584,0.065
3,국내 국채(초장기),0.218000,1.798200e-01,0.179820,0.216301,2.497500e-01,0.202532,0.205
4,글로벌 IG 크레딧,0.074000,1.198800e-01,0.119880,0.119880,9.990000e-02,0.102170,0.085
5,사모주식 PE/VC,0.047667,8.678625e-17,0.000000,0.000000,7.105254e-17,0.017784,0.030
6,실물 인프라,0.047667,7.992000e-02,0.079920,0.043439,9.990000e-02,0.062914,0.065
7,사모대출 PD,0.047667,9.990000e-02,0.099900,0.099900,2.997000e-02,0.076671,0.054
8,단기자금,0.001000,1.000000e-03,0.001000,0.001000,1.000000e-03,0.001000,0.001



 results/step11_tableC_common.csv


,Unnamed: 0,Official 2027,Official mapped,Policy MVO,Ledoit-Wolf,Box Robust,Ellipsoid Robust,Michaud mean,Team scenario
0,국내주식,0.208,0.208,0.21355,0.199672,0.206251,0.14985,0.200657,0.220
1,해외주식,0.356,0.356,0.30593,0.319808,0.313229,0.36963,0.336271,0.340
2,국내채권,0.218,0.218,0.17982,0.179820,0.216301,0.24975,0.202532,0.205
3,해외채권,0.074,0.074,0.11988,0.119880,0.119880,0.09990,0.102170,0.085
4,대체투자,0.143,0.143,0.17982,0.179820,0.143339,0.12987,0.157369,0.149
5,단기자금,0.001,0.001,0.00100,0.001000,0.001000,0.00100,0.001000,0.001



 results/step11_transition_2026H1_to_team.csv


,Unnamed: 0,2026H1_actual,2027_team,change_pp,static_balance_equivalent_trn,official_2027,official_change_pp,official_static_balance_equivalent_trn,incremental_team_vs_official_trn
0,국내주식,0.291,0.220,-7.1,-132.4576,0.208,-8.3,-154.8448,22.3872
1,해외주식,0.354,0.340,-1.4,-26.1184,0.356,0.2,3.7312,-29.8496
2,국내채권,0.154,0.205,5.1,95.1456,0.218,6.4,119.3984,-24.2528
3,해외채권,0.059,0.085,2.6,48.5056,0.074,1.5,27.9840,20.5216
4,대체투자,0.140,0.149,0.9,16.7904,0.143,0.3,5.5968,11.1936
5,단기자금,0.002,0.001,-0.1,-1.8656,0.001,-0.1,-1.8656,0.0000


,Unnamed: 0,2026H1_actual,2027_team,change_pp,static_balance_equivalent_trn,official_2027,official_change_pp,official_static_balance_equivalent_trn,incremental_team_vs_official_trn
0,국내주식,0.291,0.220,-7.1,-132.4576,0.208,-8.3,-154.8448,22.3872
1,해외주식,0.354,0.340,-1.4,-26.1184,0.356,0.2,3.7312,-29.8496
2,국내채권,0.154,0.205,5.1,95.1456,0.218,6.4,119.3984,-24.2528
3,해외채권,0.059,0.085,2.6,48.5056,0.074,1.5,27.9840,20.5216
4,대체투자,0.140,0.149,0.9,16.7904,0.143,0.3,5.5968,11.1936
5,단기자금,0.002,0.001,-0.1,-1.8656,0.001,-0.1,-1.8656,0.0000


## STEP 12 — Policy Black–Litterman

Prior는 official mapped다. δ=2.5, τ=0.025, P=I. T=10은 forward-looking CMA의 실제 표본크기가 아니라 120개월 risk sample을 이용해 Ω=Σ/T를 구현하는 **대용 가정**이며 T=5/10/20 민감도를 함께 계산한다.


In [7]:
runpy.run_path(str(ROOT/"src/step12_policy_bl.py"),run_name="__main__")
show_csv("results/step12_tableD.csv")
show_csv("results/step12_allocations.csv")
show_csv("results/step12_T_sensitivity.csv")
show_csv("results/step12_cash_sensitivity.csv")
show_csv("results/step12_TE095_sensitivity.csv")


Prior/CMA weights 0.8 0.2
Robust BL [0.19166146 0.30087354 0.02746501 0.248      0.09246858 0.01766667
 0.04319809 0.07766667]

 results/step12_tableD.csv


,asset,CMA_total,Q_excess,pi_excess,view_error_Q_minus_pi,mu_BL_excess,mu_BL_total,tauSigma_diag,Omega_diag,V_BL_diag
0,국내주식,0.062,0.032,0.039462,-0.007462,0.037969,0.067969,0.000951,0.003802,0.000760
1,글로벌 선진국 DM,0.056,0.026,0.040042,-0.014042,0.037233,0.067233,0.000656,0.002624,0.000525
2,글로벌 신흥국 EM,0.084,0.054,0.043776,0.010224,0.045821,0.075821,0.001102,0.004410,0.000882
3,국내 국채(초장기),0.036,0.006,0.005936,0.000064,0.005948,0.035948,0.000106,0.000422,0.000085
4,글로벌 IG 크레딧,0.053,0.023,0.010626,0.012374,0.013101,0.043101,0.000152,0.000608,0.000122
5,사모주식 PE/VC,0.068,0.038,0.047375,-0.009375,0.045500,0.075500,0.001210,0.004840,0.000968
6,실물 인프라,0.068,0.038,0.021166,0.016834,0.024533,0.054533,0.000302,0.001210,0.000242
7,사모대출 PD,0.078,0.048,0.016163,0.031837,0.022531,0.052531,0.000226,0.000902,0.000180



 results/step12_allocations.csv


,asset,official_mapped_total,baseline_policy_mvo,BL_MVO,Robust_BL,active_BL,active_Robust_BL
0,국내주식,0.208000,0.213778,0.185682,0.191661,-0.022318,-0.016339
1,글로벌 선진국 DM,0.327520,0.206222,0.297520,0.300874,-0.030000,-0.026646
2,글로벌 신흥국 EM,0.028480,0.100000,0.036798,0.027465,0.008318,-0.001015
3,국내 국채(초장기),0.218000,0.179000,0.238386,0.248000,0.020386,0.030000
4,글로벌 IG 크레딧,0.074000,0.120000,0.104000,0.092469,0.030000,0.018469
5,사모주식 PE/VC,0.047667,0.000000,0.017667,0.017667,-0.030000,-0.030000
6,실물 인프라,0.047667,0.080000,0.041280,0.043198,-0.006386,-0.004469
7,사모대출 PD,0.047667,0.100000,0.077667,0.077667,0.030000,0.030000



 results/step12_T_sensitivity.csv


...


 results/step12_cash_sensitivity.csv


,method,국내주식,글로벌 선진국 DM,글로벌 신흥국 EM,국내 국채(초장기),글로벌 IG 크레딧,사모주식 PE/VC,실물 인프라,사모대출 PD,단기자금,expected_return,volatility,sharpe,TE_vs_official,turnover_vs_official,utility
0,Policy_MVO_cash_optimized,0.214057,0.205943,0.100000,0.160000,0.120000,0.000000,0.080000,0.100000,0.02,0.059164,0.100682,0.289669,0.018361,0.227244,0.038891
1,BL-MVO_cash_optimized,0.185900,0.297520,0.036580,0.224363,0.094394,0.017667,0.045910,0.077667,0.02,0.056064,0.098057,0.265802,0.010000,0.083857,0.036833
2,Robust_BL_cash_optimized,0.189476,0.301598,0.028926,0.239703,0.078006,0.018448,0.046177,0.077667,0.02,0.055919,0.097766,0.265113,0.010000,0.075154,0.036803



 results/step12_TE095_sensitivity.csv


,method,국내주식,글로벌 선진국 DM,글로벌 신흥국 EM,국내 국채(초장기),글로벌 IG 크레딧,사모주식 PE/VC,실물 인프라,사모대출 PD,expected_return,volatility,sharpe,TE_vs_official,turnover_vs_official,utility
0,BL-MVO_TE095,0.187828,0.29752,0.034652,0.234114,0.098368,0.017667,0.051185,0.077667,0.056288,0.098833,0.265984,0.0095,0.080172,0.036752
1,Robust_BL_TE095,0.191727,0.30041,0.027863,0.248000,0.084539,0.018735,0.050059,0.077667,0.056155,0.098568,0.265349,0.0095,0.072931,0.036724


,method,국내주식,글로벌 선진국 DM,글로벌 신흥국 EM,국내 국채(초장기),글로벌 IG 크레딧,사모주식 PE/VC,실물 인프라,사모대출 PD,expected_return,volatility,sharpe,TE_vs_official,turnover_vs_official,utility
0,BL-MVO_TE095,0.187828,0.29752,0.034652,0.234114,0.098368,0.017667,0.051185,0.077667,0.056288,0.098833,0.265984,0.0095,0.080172,0.036752
1,Robust_BL_TE095,0.191727,0.30041,0.027863,0.248000,0.084539,0.018735,0.050059,0.077667,0.056155,0.098568,0.265349,0.0095,0.072931,0.036724


## 보조 민감도 — Box·Ellipsoid·10% 상세상한


In [8]:
runpy.run_path(str(ROOT/"src/audit_sensitivities.py"),run_name="__main__")
show_csv("results/audit_method_sensitivities.csv")
show_csv("results/audit_michaud_relaxed_caps.csv")


                   case          detail_bounds      국내주식  글로벌 선진국 DM  \
0                   MVO      10% detailed caps  0.213764    0.206236   
1                   MVO  relaxed detailed caps  0.150000    0.095125   
2           Ledoit-Wolf      10% detailed caps  0.199872    0.220128   
3           Ledoit-Wolf  relaxed detailed caps  0.150000    0.114285   
4           Box default      10% detailed caps  0.206458    0.213542   
5           Box default  relaxed detailed caps  0.150000    0.095125   
6      Box bootstrap-SE      10% detailed caps  0.150000    0.270000   
7      Box bootstrap-SE  relaxed detailed caps  0.150000    0.150412   
8   Ellipsoid bootstrap      10% detailed caps  0.150000    0.270000   
9   Ellipsoid bootstrap  relaxed detailed caps  0.150000    0.207318   
10    Ellipsoid Sigma/T      10% detailed caps  0.193356    0.272794   
11    Ellipsoid Sigma/T  relaxed detailed caps  0.194904    0.267562   

    글로벌 신흥국 EM  국내 국채(초장기)  글로벌 IG 크레딧    사모주식 PE/VC        실물 

,case,detail_bounds,국내주식,글로벌 선진국 DM,글로벌 신흥국 EM,국내 국채(초장기),글로벌 IG 크레딧,사모주식 PE/VC,실물 인프라,사모대출 PD
0,MVO,10% detailed caps,0.213764,0.206236,0.100000,0.180000,0.12000,0.000000e+00,8.000000e-02,0.10000
1,MVO,relaxed detailed caps,0.150000,0.095125,0.274875,0.180000,0.12000,4.628783e-17,0.000000e+00,0.18000
2,Ledoit-Wolf,10% detailed caps,0.199872,0.220128,0.100000,0.180000,0.12000,4.872325e-17,8.000000e-02,0.10000
3,Ledoit-Wolf,relaxed detailed caps,0.150000,0.114285,0.255715,0.180000,0.12000,0.000000e+00,2.262080e-18,0.18000
4,Box default,10% detailed caps,0.206458,0.213542,0.100000,0.216517,0.12000,8.736736e-18,4.348287e-02,0.10000
5,Box default,relaxed detailed caps,0.150000,0.095125,0.274875,0.180000,0.12000,0.000000e+00,6.133174e-19,0.18000
6,Box bootstrap-SE,10% detailed caps,0.150000,0.270000,0.100000,0.250000,0.10000,6.265534e-18,1.000000e-01,0.03000
7,Box bootstrap-SE,relaxed detailed caps,0.150000,0.150412,0.219588,0.250000,0.10000,6.409143e-17,1.300000e-01,0.00000
8,Ellipsoid bootstrap,10% detailed caps,0.150000,0.270000,0.100000,0.250000,0.10000,0.000000e+00,1.000000e-01,0.03000
9,Ellipsoid bootstrap,relaxed detailed caps,0.150000,0.207318,0.162682,0.250000,0.10000,0.000000e+00,1.058296e-01,0.02417



 results/audit_michaud_relaxed_caps.csv


,asset,mean_relaxed,p05_relaxed,p50_relaxed,p95_relaxed
0,국내주식,0.191261,0.15,1.500000e-01,0.250000
1,글로벌 선진국 DM,0.136039,0.00,8.245712e-02,0.370000
2,글로벌 신흥국 EM,0.211695,0.00,2.760080e-01,0.380503
3,국내 국채(초장기),0.195080,0.15,1.800000e-01,0.250000
4,글로벌 IG 크레딧,0.098383,0.05,1.200000e-01,0.120000
5,사모주식 PE/VC,0.009396,0.00,0.000000e+00,0.073358
6,실물 인프라,0.044859,0.00,4.493271e-17,0.180000
7,사모대출 PD,0.113287,0.00,1.800000e-01,0.180000


,asset,mean_relaxed,p05_relaxed,p50_relaxed,p95_relaxed
0,국내주식,0.191261,0.15,1.500000e-01,0.250000
1,글로벌 선진국 DM,0.136039,0.00,8.245712e-02,0.370000
2,글로벌 신흥국 EM,0.211695,0.00,2.760080e-01,0.380503
3,국내 국채(초장기),0.195080,0.15,1.800000e-01,0.250000
4,글로벌 IG 크레딧,0.098383,0.05,1.200000e-01,0.120000
5,사모주식 PE/VC,0.009396,0.00,0.000000e+00,0.073358
6,실물 인프라,0.044859,0.00,4.493271e-17,0.180000
7,사모대출 PD,0.113287,0.00,1.800000e-01,0.180000


## STEP 13 — Stress 1~3

1. **과제 정의 Stress 1:** Robust BL active 방향에 불리한 CMA 직접 1SE 실패, `SE=√diag(Ω)`, fixed weights. 이 기준에서는 Robust BL이 가장 불리하게 나오는 결과를 그대로 보고한다.
2. **보조 민감도:** 각 후보를 자기 active 방향에 불리하게 같은 1SE로 충격한다. 이는 과제 Stress 1을 대체하지 않고 각 안 자신의 베팅 실패를 비교하기 위한 진단이다.
3. Stress 2: 정책 주식(국내/DM/EM) -2%p 및 상관 +0.15. PE 포함은 sensitivity.
4. Stress 3: 대체 σ×1.5, 대체-글로벌주식 상관 +0.20. eigen-clipping과 Higham을 모두 보고한다.


In [9]:
runpy.run_path(str(ROOT/"src/step13_stress_test.py"),run_name="__main__")
show_csv("results/step13_stress1_direct_cma.csv")
show_csv("results/step13_stress1_self_active_sensitivity.csv")
show_csv("results/step13_stress_results.csv")
show_csv("results/step13_covariance_diagnostics.csv")
show_csv("results/step13_T_stress1_sensitivity.csv")


   portfolio  baseline_expected_return  stress_expected_return  \
0   Official                  0.055630                0.084362   
1       Team                  0.057087                0.085473   
2  Robust_BL                  0.055095                0.077171   

   expected_return_change  baseline_active_return_vs_official  \
0                0.028732                            0.000000   
1                0.028386                            0.001457   
2                0.022076                           -0.000535   

   stress_active_return_vs_official  active_return_change  \
0                          0.000000              0.000000   
1                          0.001111             -0.000346   
2                         -0.007191             -0.006656   

   baseline_volatility  stress_volatility  baseline_utility  stress_utility  \
0             0.107067           0.107067          0.032703        0.061435   
1             0.106071           0.106071          0.034585        0.06

,portfolio,baseline_expected_return,stress_expected_return,expected_return_change,baseline_active_return_vs_official,stress_active_return_vs_official,active_return_change,baseline_volatility,stress_volatility,baseline_utility,stress_utility,utility_change
0,Official,0.055630,0.084362,0.028732,0.000000,0.000000,0.000000,0.107067,0.107067,0.032703,0.061435,0.028732
1,Team,0.057087,0.085473,0.028386,0.001457,0.001111,-0.000346,0.106071,0.106071,0.034585,0.062971,0.028386
2,Robust_BL,0.055095,0.077171,0.022076,-0.000535,-0.007191,-0.006656,0.098159,0.098159,0.035824,0.057901,0.022076



 results/step13_stress1_self_active_sensitivity.csv


,portfolio,baseline_expected_return,stress_expected_return,expected_return_change,baseline_active_return_vs_official,stress_active_return_vs_official,active_return_change,baseline_volatility,stress_volatility,baseline_utility,stress_utility,utility_change
0,Team,0.057087,0.053613,-0.003474,0.001457,-0.006960,-0.008417,0.106071,0.106071,0.034585,0.031111,-0.003474
1,Robust_BL,0.055095,0.077171,0.022076,-0.000535,-0.007191,-0.006656,0.098159,0.098159,0.035824,0.057901,0.022076
2,Blend_50_RBL,0.056091,0.071418,0.015327,0.000461,-0.005846,-0.006306,0.102028,0.102028,0.035271,0.050598,0.015327



 results/step13_stress_results.csv


...


 results/step13_covariance_diagnostics.csv


,scenario,raw_min_corr_eigenvalue,correction,final_min_corr_eigenvalue
0,Stress2_policy_equity,0.033518,none,3.351825e-02
1,Stress2_include_PE,-0.042749,Higham,-9.128217e-13
2,Stress3,-0.200766,eigenvalue clipping,9.455656e-09
3,Stress3,-0.200766,Higham,-9.632922e-13



 results/step13_T_stress1_sensitivity.csv


,T_years,portfolio,baseline_expected_return,stress_expected_return,expected_return_change,baseline_active_return_vs_official,stress_active_return_vs_official,active_return_change
0,5.0,Team,0.057087,0.097231,0.040144,0.001457,0.000968,-0.000489
1,5.0,Robust_BL,0.055042,0.086270,0.031228,-0.000588,-0.009994,-0.009405
2,10.0,Team,0.057087,0.085473,0.028386,0.001457,0.001111,-0.000346
3,10.0,Robust_BL,0.055095,0.077171,0.022076,-0.000535,-0.007191,-0.006656
4,20.0,Team,0.057087,0.071055,0.013968,0.001457,-0.002217,-0.003674
5,20.0,Robust_BL,0.055196,0.067847,0.012651,-0.000434,-0.005425,-0.004991


,T_years,portfolio,baseline_expected_return,stress_expected_return,expected_return_change,baseline_active_return_vs_official,stress_active_return_vs_official,active_return_change
0,5.0,Team,0.057087,0.097231,0.040144,0.001457,0.000968,-0.000489
1,5.0,Robust_BL,0.055042,0.086270,0.031228,-0.000588,-0.009994,-0.009405
2,10.0,Team,0.057087,0.085473,0.028386,0.001457,0.001111,-0.000346
3,10.0,Robust_BL,0.055095,0.077171,0.022076,-0.000535,-0.007191,-0.006656
4,20.0,Team,0.057087,0.071055,0.013968,0.001457,-0.002217,-0.003674
5,20.0,Robust_BL,0.055196,0.067847,0.012651,-0.000434,-0.005425,-0.004991


## STEP 14 — Robust BL 최종 의결

최종안은 **Robust BL (TE 1.0%)**이다. 과제 정의 Stress 1에서는 BL을 직접 겨냥하므로 Robust BL이 가장 불리하지만, 자기 active 방향 보조 민감도와 Stress 2·3에서는 Team보다 손실이 작다. CMA 기준 기대수익률은 공식보다 약 5bp 낮지만 변동성을 약 0.9%p 낮춘다. 목표비중 기준 사전 TE는 1.00%로 한도 내 경계이며, 국내채권 +3%p·PE -3%p·PD +3%p가 active 경계에 붙는다. `step14_decision_aid.csv`의 손익분기 확률은 감사 참고용으로만 남기고 최종 선택 근거에서는 제외한다.


In [10]:
runpy.run_path(str(ROOT/"src/step14_candidate_comparison.py"),run_name="__main__")
show_csv("results/step14_candidate_comparison.csv")
runpy.run_path(str(ROOT/"src/step14_final_decision.py"),run_name="__main__")
show_csv("results/step14_final_weights.csv")
show_csv("submission/team2_views.csv")
print((RESULTS/"step14_final_resolution.json").read_text(encoding="utf-8"))


          candidate           decision_role  expected_return  \
0              Team  comparison_alternative         0.057087   
1  Robust BL TE1.00          selected_final         0.055095   
2  Robust BL TE0.95             sensitivity         0.055225   
3     Blend 25% RBL             sensitivity         0.056589   
4     Blend 50% RBL  comparison_alternative         0.056091   
5     Blend 75% RBL             sensitivity         0.055593   

   expected_return_vs_official  volatility_common_Sigma   utility        TE  \
0                     0.001457                 0.106071  0.034585  0.008625   
1                    -0.000535                 0.098159  0.035824  0.010000   
2                    -0.000405                 0.098568  0.035794  0.009500   
3                     0.000959                 0.104029  0.034945  0.007472   
4                     0.000461                 0.102028  0.035271  0.007342   
5                    -0.000037                 0.100071  0.035564  0.008284  

,candidate,국내주식,글로벌 선진국 DM,글로벌 신흥국 EM,국내 국채(초장기),글로벌 IG 크레딧,사모주식 PE/VC,실물 인프라,사모대출 PD,expected_return,...,turnover,stress1_expected_return_change,stress1_active_loss,stress1_utility_change,stress2_utility_change,stress3_Higham_utility_change,domestic_equity_full_sell_trn,domestic_equity_incremental_vs_official_trn,stress1_self_active_loss,decision_role
0,Team,0.220000,0.275000,0.065000,0.20500,0.085000,0.030000,0.065000,0.054000,0.057087,...,0.083187,0.028386,-0.000346,0.028386,-0.013063,-0.003641,132.457600,-22.387200,-0.008417,comparison_alternative
1,Robust BL TE1.00,0.191661,0.300874,0.027465,0.24800,0.092469,0.017667,0.043198,0.077667,0.055095,...,0.078469,0.022076,-0.006656,0.022076,-0.011791,-0.002733,185.325986,30.481186,-0.006656,selected_final
2,Robust BL TE0.95,0.191727,0.300410,0.027863,0.24800,0.084539,0.018735,0.050059,0.077667,0.055225,...,0.072931,0.022591,-0.006141,0.022591,-0.011794,-0.002953,185.203297,30.358497,NaN,sensitivity
3,Blend 25% RBL,0.212915,0.281468,0.055616,0.21575,0.086867,0.026917,0.059550,0.059917,0.056589,...,0.069052,0.026809,-0.001923,0.026809,-0.012746,-0.003403,145.674696,-9.170104,NaN,sensitivity
4,Blend 50% RBL,0.205831,0.287937,0.046233,0.22650,0.088734,0.023833,0.054099,0.065833,0.056091,...,0.065586,0.025231,-0.003501,0.025231,-0.012429,-0.003173,158.891793,4.046993,-0.006306,comparison_alternative
5,Blend 75% RBL,0.198746,0.294405,0.036849,0.23725,0.090601,0.020750,0.048649,0.071750,0.055593,...,0.069285,0.023654,-0.005078,0.023654,-0.012110,-0.002950,172.108889,17.264089,NaN,sensitivity


위원회는 2027년 목표비중을 수정한다. 수정폭은 자산별로 국내주식 -1.6%p, 해외주식 -2.8%p, 국내채권 +3.0%p, 해외채권 +1.8%p, 대체투자 -0.4%p, 단기자금 0.0%p다. 이 결정은 기준 CMA, 공분산 추정, CMA 절대수익률 뷰와 데이터 기반 뷰 불확실성에 근거한다. 공식 비중을 사전비중으로 둔 Robust BL은 CMA 기준 기대수익률이 공식보다 약 5bp 낮지만 변동성을 0.9%p 낮춘다. 다만 운용 중 실제 비중 드리프트로 사전 TE가 1.0%를 넘으면 리밸런싱하고, 리밸런싱 후에도 초과하거나 최근 8분기 뷰 적중률이 55% 미만이거나 공식 경로 대비 추가 국내주식 거래가 ADV의 1%를 넘으면 공식 목표로 복귀하거나 재심의한다.
       level       asset  official_or_mapped  team_2027    BL_MVO  Robust_BL  \
0   detailed        국내주식            0.208000      0.220  0.185682   0.191661   
1   detailed  글로벌 선진국 DM            0.327520      0.275  0.297520   0.300874   
2   detailed  글로벌 신흥국 EM            0.028480      0.065  0.036798   0.027465   
3   detailed  국내 국채(초장기)            0.218000      0.205  0.238386   0.248000   
4   detailed  글로벌 IG 크레딧            0.074000      0.085  0.104000   0.092469   
5   detailed  사모주식 PE/VC            0.047667      0.030  0.017667   0.017667   
6   detailed      실물 인프라            0.047667      0.065  0.0412

,level,asset,official_or_mapped,team_2027,BL_MVO,Robust_BL,Blend_50_RBL,final_decision,delta_final_vs_official
0,detailed,국내주식,0.208000,0.220,0.185682,0.191661,0.205831,0.191661,-0.016339
1,detailed,글로벌 선진국 DM,0.327520,0.275,0.297520,0.300874,0.287937,0.300874,-0.026646
2,detailed,글로벌 신흥국 EM,0.028480,0.065,0.036798,0.027465,0.046233,0.027465,-0.001015
3,detailed,국내 국채(초장기),0.218000,0.205,0.238386,0.248000,0.226500,0.248000,0.030000
4,detailed,글로벌 IG 크레딧,0.074000,0.085,0.104000,0.092469,0.088734,0.092469,0.018469
5,detailed,사모주식 PE/VC,0.047667,0.030,0.017667,0.017667,0.023833,0.017667,-0.030000
6,detailed,실물 인프라,0.047667,0.065,0.041280,0.043198,0.054099,0.043198,-0.004469
7,detailed,사모대출 PD,0.047667,0.054,0.077667,0.077667,0.065833,0.077667,0.030000
8,common,국내주식,0.208000,0.220,0.185682,0.191661,0.205831,0.191661,-0.016339
9,common,해외주식,0.356000,0.340,0.334318,0.328339,0.334169,0.328339,-0.027661



 submission/team2_views.csv


,asset,CMA_total,Q_excess,pi_excess,view_error_Q_minus_pi,mu_BL_total,Omega_diag,V_BL_diag
0,국내주식,0.062,0.032,0.039462,-0.007462,0.067969,0.003802,0.000760
1,글로벌 선진국 DM,0.056,0.026,0.040042,-0.014042,0.067233,0.002624,0.000525
2,글로벌 신흥국 EM,0.084,0.054,0.043776,0.010224,0.075821,0.004410,0.000882
3,국내 국채(초장기),0.036,0.006,0.005936,0.000064,0.035948,0.000422,0.000085
4,글로벌 IG 크레딧,0.053,0.023,0.010626,0.012374,0.043101,0.000608,0.000122
5,사모주식 PE/VC,0.068,0.038,0.047375,-0.009375,0.075500,0.004840,0.000968
6,실물 인프라,0.068,0.038,0.021166,0.016834,0.054533,0.001210,0.000242
7,사모대출 PD,0.078,0.048,0.016163,0.031837,0.052531,0.000902,0.000180


{
  "status": "DECIDED",
  "decision": "Robust BL TE1.00",
  "final_common_weights": {
    "국내주식": 0.1916614569945745,
    "해외주식": 0.32833854300542537,
    "국내채권": 0.248,
    "해외채권": 0.0924685766698695,
    "대체투자": 0.1385314233301303,
    "단기자금": 0.001
  },
  "diagnostics": {
    "expected_return": 0.0550946269628761,
    "expected_return_vs_official": -0.0005354797037904,
    "volatility_common_Sigma": 0.0981585672002656,
    "utility": 0.035824418333258,
    "TE": 0.01,
    "turnover": 0.0784685766698695,
    "stress1_active_loss": -0.0066559569540714,
    "stress2_utility_change": -0.0117910092212874,
    "stress3_Higham_utility_change": -0.0027332290841474
  },
  "binding_constraints": {
    "국내채권_active_pp": 3.0,
    "사모주식_PE_active_pp": -3.0,
    "사모대출_PD_active_pp": 3.0,
    "target_weight_ex_ante_TE_pct": 1.0
  },
  "implementation": {
    "2026H1_to_final_domestic_equity_sell_trn": 185.32598583092178,
    "incremental_vs_official_domestic_equity_sell_trn": 30.48118583092182,
 